# 04. 가중 거리 기반 상권 추천 엔진

최근 4분기 구조 프로필로 사용자 조건 적합도를 계산하고, 2021Q1~2025Q4의 관측 evidence를 신뢰도 보정해 최종 점수를 만든다. 미래 매출 예측, KNN 분류·회귀, GPT 생성 설명은 사용하지 않는다.

`final_score = 0.60 × condition_fit_score + 0.40 × reliability_adjusted_evidence_score`

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.area_recommender import (
    AreaRecommender, DEFAULT_K, RecommendationRequest,
    build_recommendation_index, feature_mapping_table,
    load_recommender_inputs, save_recommender_artifacts,
    scenario_suite, score_weights_table,
)
PROJECT_ROOT

## 최근 4분기 추천 인덱스

area profile의 2025Q1~Q4 평균을 사용한다. identifier, 이름, 품질 플래그는 거리 피처가 아니며 매출 관련 컬럼은 포함하지 않는다. 8분기 구조 추세는 보조 컬럼으로 보존한다.

In [ ]:
area_profile, evidence, profile_dictionary, evidence_dictionary = load_recommender_inputs(PROJECT_ROOT)
recommendation_index = build_recommendation_index(area_profile, profile_dictionary)
recommender = AreaRecommender(recommendation_index, evidence)
pd.Series({
    'areas': recommendation_index['area_code'].nunique(),
    'rows': len(recommendation_index),
    'columns': len(recommendation_index.columns),
    'area_duplicates': int(recommendation_index['area_code'].duplicated().sum()),
    'sales_like_columns': [c for c in recommendation_index if 'sales' in c.lower() or '매출' in c],
})

In [ ]:
display(feature_mapping_table())
display(score_weights_table())

## 예시 사용자 시나리오

실제 매출 업종코드 5개를 사용한다. 지정하지 않은 조건에는 선호값을 부여하지 않는다.

In [ ]:
scenarios = {
    '01_cafe_young_weekend_evening': RecommendationRequest(
        industry_code='CS100010', target_age_groups=('20',),
        preferred_time_bands=('17_21', '21_24'), weekend_importance=0.9,
        floating_population_importance=0.9, top_n=10,
    ),
    '02_korean_office_lunch': RecommendationRequest(
        industry_code='CS100001', target_age_groups=('30', '40'),
        preferred_time_bands=('11_14',), worker_population_importance=1.0,
        floating_population_importance=0.4, top_n=10,
    ),
    '03_convenience_residential': RecommendationRequest(
        industry_code='CS300002', preferred_area_types=('A', 'D', 'R'),
        resident_population_importance=1.0, apartment_importance=1.0,
        store_density_preference='high', top_n=10,
    ),
    '04_academy_students_residential': RecommendationRequest(
        industry_code='CS200001', preferred_area_types=('A', 'D'),
        target_age_groups=('10', '20'), education_facility_importance=1.0,
        resident_population_importance=0.8, apartment_importance=0.5, top_n=10,
    ),
    '05_beauty_women_20_40': RecommendationRequest(
        industry_code='CS200028', target_gender='female',
        target_age_groups=('20', '30', '40'), floating_population_importance=0.8,
        resident_population_importance=0.7, apartment_importance=0.6, top_n=10,
    ),
}
scenarios

In [ ]:
sample_results, weight_sensitivity, k_stability, validation = scenario_suite(recommender, scenarios)
save_recommender_artifacts(
    recommendation_index, sample_results, weight_sensitivity, k_stability, validation,
    project_root=PROJECT_ROOT,
)
sample_results.shape, validation['status'].value_counts().to_dict()

In [ ]:
sample_results[['scenario', 'rank', 'industry_name', 'area_code', 'area_name', 'district_name', 'final_score', 'condition_fit_score', 'raw_evidence_score', 'reliability_adjusted_evidence_score', 'reliability_grade']].groupby('scenario').head(10)

## 안정성과 방법 선택

weighted Euclidean은 high/low 방향 선호가 단조롭게 반영되고 기여도를 직접 설명할 수 있어 기본 거리로 선택한다. k=10·20·30·50을 비교하며, evidence 40%가 후보 집합에서 충분히 작동하도록 기본 k=50을 사용한다.

In [ ]:
weight_sensitivity.groupby('scenario').agg(
    overlap_min=('top10_overlap', 'min'), overlap_mean=('top10_overlap', 'mean'),
    correlation_min=('top10_rank_correlation', 'min'), correlation_mean=('top10_rank_correlation', 'mean'),
)

In [ ]:
k_stability

In [ ]:
validation

산출물:

- `data/processed/area_recommendation_index.parquet`
- `outputs/tables/recommender_feature_mapping.csv`
- `outputs/tables/recommender_score_weights.csv`
- `outputs/tables/recommender_validation.csv`
- `outputs/tables/recommender_sample_results.csv`
- `outputs/tables/recommender_weight_sensitivity.csv`
- `outputs/tables/recommender_k_stability.csv`